# Exercise 2 - Lossless Audio Compression with Rice Coding (`.ex2`)

This notebook implements a custom **Rice encoder/decoder** for WAV audio files.

## Deliverables covered
- Encode `Sound1.wav` and `Sound2.wav` into `.ex2` using Rice coding.
- Decode `.ex2` back into WAV files.
- Prove losslessness using `np.array_equal(original, decoded)`.
- Compare compression performance for `K=4` and `K=2`.
- Provide a short report (~500 words) with analysis and discussion.

In [8]:
%pip install -q numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from pathlib import Path
import struct
import wave
import numpy as np

BASE_DIR = Path(".")
INPUT_FILES = [BASE_DIR / "Sound1.wav", BASE_DIR / "Sound2.wav"]
K_VALUES = [4, 2]

for input_path in INPUT_FILES:
    assert input_path.exists(), f"Missing input file: {input_path.resolve()}"

print("Input files:")
for input_path in INPUT_FILES:
    print("-", input_path.name)

Input files:
- Sound1.wav
- Sound2.wav


In [10]:
class BitWriter:
    def __init__(self):
        self.buffer = bytearray()
        self.current_byte = 0
        self.bit_count = 0

    def _flush_if_full(self):
        if self.bit_count == 8:
            self.buffer.append(self.current_byte)
            self.current_byte = 0
            self.bit_count = 0

    def write_bit(self, bit):
        self.current_byte = (self.current_byte << 1) | (1 if bit else 0)
        self.bit_count += 1
        self._flush_if_full()

    def write_bits(self, value, num_bits):
        if num_bits <= 0:
            return

        remaining = num_bits
        while remaining > 0:
            space = 8 - self.bit_count
            take = remaining if remaining < space else space
            shift = remaining - take
            chunk = (value >> shift) & ((1 << take) - 1)

            self.current_byte = (self.current_byte << take) | chunk
            self.bit_count += take
            remaining -= take
            self._flush_if_full()

    def write_unary(self, q):
        # Emit q ones, then one terminating zero.
        while q > 0:
            space = 8 - self.bit_count
            take = q if q < space else space
            self.current_byte = (self.current_byte << take) | ((1 << take) - 1)
            self.bit_count += take
            q -= take
            self._flush_if_full()

        self.current_byte <<= 1
        self.bit_count += 1
        self._flush_if_full()

    def to_bytes(self):
        padding_bits = 0
        if self.bit_count > 0:
            padding_bits = 8 - self.bit_count
            self.current_byte <<= padding_bits
            self.buffer.append(self.current_byte)
        return bytes(self.buffer), padding_bits


class BitReader:
    def __init__(self, data):
        self.data = memoryview(data)
        self.byte_index = 0
        self.bit_buffer = 0
        self.bits_left = 0

    def _fill(self, min_bits):
        n = len(self.data)
        while self.bits_left < min_bits and self.byte_index < n:
            self.bit_buffer = (self.bit_buffer << 8) | self.data[self.byte_index]
            self.bits_left += 8
            self.byte_index += 1

    def read_bits(self, num_bits):
        if num_bits == 0:
            return 0

        self._fill(num_bits)
        if self.bits_left < num_bits:
            raise EOFError("Unexpected end of bitstream")

        shift = self.bits_left - num_bits
        value = (self.bit_buffer >> shift) & ((1 << num_bits) - 1)

        self.bits_left -= num_bits
        if self.bits_left:
            self.bit_buffer &= (1 << self.bits_left) - 1
        else:
            self.bit_buffer = 0

        return value

    def read_unary(self):
        q = 0

        while True:
            if self.bits_left == 0:
                if self.byte_index >= len(self.data):
                    raise EOFError("Unexpected end of bitstream")
                self.bit_buffer = self.data[self.byte_index]
                self.bits_left = 8
                self.byte_index += 1

            # If all remaining buffered bits are 1s, consume them in one step.
            all_ones_mask = (1 << self.bits_left) - 1
            if self.bit_buffer == all_ones_mask:
                q += self.bits_left
                self.bit_buffer = 0
                self.bits_left = 0
                continue

            inv = (~self.bit_buffer) & all_ones_mask
            ones_before_zero = self.bits_left - inv.bit_length()

            q += ones_before_zero
            consume = ones_before_zero + 1
            self.bits_left -= consume
            if self.bits_left:
                self.bit_buffer &= (1 << self.bits_left) - 1
            else:
                self.bit_buffer = 0
            return q


def zigzag_encode(samples):
    s = samples.astype(np.int64)
    return np.where(s >= 0, s * 2, -2 * s - 1).astype(np.uint64)


def zigzag_decode(values):
    v = values.astype(np.uint64)
    decoded = np.where((v & 1) == 0, v // 2, -((v + 1) // 2))
    return decoded.astype(np.int64)


def encode_rice(values_unsigned, k):
    writer = BitWriter()
    remainder_mask = (1 << k) - 1

    for value in values_unsigned:
        value_int = int(value)
        q = value_int >> k
        r = value_int & remainder_mask

        writer.write_unary(q)
        writer.write_bits(r, k)

    payload, padding_bits = writer.to_bytes()
    return payload, padding_bits


def decode_rice(payload, k, sample_count):
    reader = BitReader(payload)
    values = np.empty(sample_count, dtype=np.uint64)

    for i in range(sample_count):
        q = reader.read_unary()
        r = reader.read_bits(k)
        values[i] = (q << k) | r

    return values


def read_wav_pcm_signed(path):
    with wave.open(str(path), "rb") as wf:
        params = wf.getparams()
        nchannels = wf.getnchannels()
        sampwidth = wf.getsampwidth()
        framerate = wf.getframerate()
        nframes = wf.getnframes()
        raw_bytes = wf.readframes(nframes)

    if sampwidth == 1:
        raw = np.frombuffer(raw_bytes, dtype=np.uint8)
        signed = raw.astype(np.int16) - 128
        signed_dtype = np.int16
    elif sampwidth == 2:
        signed = np.frombuffer(raw_bytes, dtype="<i2").astype(np.int32)
        signed_dtype = np.int32
    elif sampwidth == 4:
        signed = np.frombuffer(raw_bytes, dtype="<i4").astype(np.int64)
        signed_dtype = np.int64
    else:
        raise ValueError(f"Unsupported sample width: {sampwidth} bytes")

    metadata = {
        "nchannels": nchannels,
        "sampwidth": sampwidth,
        "framerate": framerate,
        "nframes": nframes,
        "signed_dtype": str(np.dtype(signed_dtype)),
        "raw_bytes": raw_bytes,
        "params": params,
    }
    return signed, metadata


def signed_to_wav_bytes(samples_signed, sampwidth):
    s = samples_signed.astype(np.int64)

    if sampwidth == 1:
        u8 = np.clip(s + 128, 0, 255).astype(np.uint8)
        return u8.tobytes()
    if sampwidth == 2:
        i16 = np.clip(s, -32768, 32767).astype("<i2")
        return i16.tobytes()
    if sampwidth == 4:
        i32 = np.clip(s, -2147483648, 2147483647).astype("<i4")
        return i32.tobytes()

    raise ValueError(f"Unsupported sample width: {sampwidth} bytes")


def write_wav_from_signed(path, samples_signed, metadata):
    raw = signed_to_wav_bytes(samples_signed, metadata["sampwidth"])
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(metadata["nchannels"])
        wf.setsampwidth(metadata["sampwidth"])
        wf.setframerate(metadata["framerate"])
        wf.writeframes(raw)


def write_ex2(path, k, metadata, sample_count, padding_bits, payload):
    header = struct.pack(
        "<4sBBHIII",
        b"EX2R",
        1,
        k,
        metadata["nchannels"],
        metadata["sampwidth"],
        metadata["framerate"],
        sample_count,
    )
    trailer = struct.pack("<B", padding_bits)
    with open(path, "wb") as f:
        f.write(header)
        f.write(payload)
        f.write(trailer)


def read_ex2(path):
    with open(path, "rb") as f:
        raw = f.read()

    if len(raw) < 4 + 1 + 1 + 2 + 4 + 4 + 1:
        raise ValueError("Invalid EX2 file: too short")

    magic, version, k, nchannels, sampwidth, framerate, sample_count = struct.unpack(
        "<4sBBHIII", raw[:20]
    )
    if magic != b"EX2R":
        raise ValueError("Invalid EX2 file: wrong magic")
    if version != 1:
        raise ValueError(f"Unsupported EX2 version: {version}")

    padding_bits = raw[-1]
    payload = raw[20:-1]

    return {
        "k": k,
        "nchannels": nchannels,
        "sampwidth": sampwidth,
        "framerate": framerate,
        "sample_count": sample_count,
        "padding_bits": padding_bits,
        "payload": payload,
    }


def encode_wav_to_ex2(input_wav, output_ex2, k):
    samples_signed, metadata = read_wav_pcm_signed(input_wav)
    mapped = zigzag_encode(samples_signed)
    payload, padding_bits = encode_rice(mapped, k)
    write_ex2(output_ex2, k, metadata, len(mapped), padding_bits, payload)
    return samples_signed, metadata


def decode_ex2_to_wav(input_ex2, output_wav):
    ex2 = read_ex2(input_ex2)
    decoded_unsigned = decode_rice(ex2["payload"], ex2["k"], ex2["sample_count"])
    decoded_signed = zigzag_decode(decoded_unsigned)

    metadata = {
        "nchannels": ex2["nchannels"],
        "sampwidth": ex2["sampwidth"],
        "framerate": ex2["framerate"],
    }
    write_wav_from_signed(output_wav, decoded_signed, metadata)
    return decoded_signed, ex2


def file_size_bytes(path):
    return Path(path).stat().st_size

In [11]:
def to_residual(samples_signed):
    s = samples_signed.astype(np.int64)
    residual = np.empty_like(s, dtype=np.int64)
    residual[0] = s[0]
    residual[1:] = s[1:] - s[:-1]
    return residual


def from_residual(residual):
    return np.cumsum(residual.astype(np.int64), dtype=np.int64)


def encode_wav_to_ex2(input_wav, output_ex2, k):
    samples_signed, metadata = read_wav_pcm_signed(input_wav)
    residual = to_residual(samples_signed)
    mapped = zigzag_encode(residual)
    payload, padding_bits = encode_rice(mapped, k)
    write_ex2(output_ex2, k, metadata, len(mapped), padding_bits, payload)
    return samples_signed, metadata


def decode_ex2_to_wav(input_ex2, output_wav):
    ex2 = read_ex2(input_ex2)
    decoded_unsigned = decode_rice(ex2["payload"], ex2["k"], ex2["sample_count"])
    decoded_residual = zigzag_decode(decoded_unsigned)
    decoded_signed = from_residual(decoded_residual)

    metadata = {
        "nchannels": ex2["nchannels"],
        "sampwidth": ex2["sampwidth"],
        "framerate": ex2["framerate"],
    }
    write_wav_from_signed(output_wav, decoded_signed, metadata)
    return decoded_signed, ex2


print("Using predictive Rice coding: first-order residuals + Rice entropy coding")

Using predictive Rice coding: first-order residuals + Rice entropy coding


In [12]:
test_values = np.array([-6, -3, -1, 0, 1, 2, 5, 9], dtype=np.int32)
print("Small integer test values:", test_values.tolist())

for k in K_VALUES:
    mapped = zigzag_encode(test_values)
    payload, _ = encode_rice(mapped, k)
    recovered_unsigned = decode_rice(payload, k, len(mapped))
    recovered_signed = zigzag_decode(recovered_unsigned).astype(np.int32)
    ok = np.array_equal(test_values, recovered_signed)
    print(f"K={k}: small-integer roundtrip ok -> {ok}")

assert ok, "Small integer sanity test failed."

Small integer test values: [-6, -3, -1, 0, 1, 2, 5, 9]
K=4: small-integer roundtrip ok -> True
K=2: small-integer roundtrip ok -> True


In [13]:
run_rows = []

for wav_path in INPUT_FILES:
    original_size = file_size_bytes(wav_path)

    for k in K_VALUES:
        ex2_path = wav_path.with_name(f"{wav_path.stem}_Enc_k{k}.ex2")
        dec_wav_path = wav_path.with_name(f"{wav_path.stem}_Enc_Dec_k{k}.wav")

        original_signed, metadata = encode_wav_to_ex2(wav_path, ex2_path, k)
        decoded_signed, ex2_info = decode_ex2_to_wav(ex2_path, dec_wav_path)

        raw_decoded = dec_wav_path.read_bytes()[44:]
        raw_original = metadata["raw_bytes"]

        lossless_samples = np.array_equal(original_signed, decoded_signed)
        lossless_raw = raw_original == raw_decoded

        encoded_size = file_size_bytes(ex2_path)
        compression_pct = (1 - (encoded_size / original_size)) * 100

        run_rows.append({
            "sound": wav_path.name,
            "k": k,
            "original_size": original_size,
            "encoded_size": encoded_size,
            "compression_pct": compression_pct,
            "lossless_samples": lossless_samples,
            "lossless_raw": lossless_raw,
            "ex2": ex2_path.name,
            "decoded_wav": dec_wav_path.name,
        })

        print(
            f"{wav_path.name} | K={k} | lossless(samples)={lossless_samples} | "
            f"lossless(raw)={lossless_raw} | encoded={encoded_size} B"
        )

all_lossless = all(r["lossless_samples"] and r["lossless_raw"] for r in run_rows)
print("\nOverall lossless status:", all_lossless)
assert all_lossless, "Decoded output differs from original; lossless requirement not met."

Sound1.wav | K=4 | lossless(samples)=True | lossless(raw)=True | encoded=857472 B
Sound1.wav | K=2 | lossless(samples)=True | lossless(raw)=True | encoded=2429926 B
Sound2.wav | K=4 | lossless(samples)=True | lossless(raw)=True | encoded=56793309 B
Sound2.wav | K=2 | lossless(samples)=True | lossless(raw)=True | encoded=226196628 B

Overall lossless status: True


In [14]:
summary = {}
for r in run_rows:
    sound = r["sound"]
    if sound not in summary:
        summary[sound] = {"original_size": r["original_size"]}
    summary[sound][f"k{r['k']}_size"] = r["encoded_size"]
    summary[sound][f"k{r['k']}_pct"] = r["compression_pct"]

print("Compression table")
print(f"{'Sound':<14} {'Original size (B)':<18} {'Rice K=4 (B)':<14} {'Rice K=2 (B)':<14} {'% Comp K=4':<12} {'% Comp K=2':<12}")
print("-" * 88)
for sound in sorted(summary.keys()):
    row = summary[sound]
    print(
        f"{sound:<14} {row['original_size']:<18} {row.get('k4_size', 0):<14} {row.get('k2_size', 0):<14} "
        f"{row.get('k4_pct', 0):<12.2f} {row.get('k2_pct', 0):<12.2f}"
    )

print("\nCreated files:")
for r in run_rows:
    print(f"- {r['ex2']}  |  {r['decoded_wav']}")

Compression table
Sound          Original size (B)  Rice K=4 (B)   Rice K=2 (B)   % Comp K=4   % Comp K=2  
----------------------------------------------------------------------------------------
Sound1.wav     1002088            857472         2429926        14.43        -142.49     
Sound2.wav     1008044            56793309       226196628      -5534.01     -22339.16   

Created files:
- Sound1_Enc_k4.ex2  |  Sound1_Enc_Dec_k4.wav
- Sound1_Enc_k2.ex2  |  Sound1_Enc_Dec_k2.wav
- Sound2_Enc_k4.ex2  |  Sound2_Enc_Dec_k4.wav
- Sound2_Enc_k2.ex2  |  Sound2_Enc_Dec_k2.wav
